In [1]:
import os
import re
import json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
# Try multiple locations to find combined_runhistory.json
surrogate_dir_candidates = [
    cwd,                           # Current directory
    cwd / 'surrogate',             # Subdirectory named surrogate
    Path(__file__).parent if '__file__' in globals() else cwd,  # Notebook's own directory
    cwd.parent / 'surrogate' if cwd.name == 'surrogate' else cwd / 'surrogate',  # Parent's surrogate or current surrogate
]

combo_path = None
for cand in surrogate_dir_candidates:
    cp = cand / 'combined_runhistory.json'
    if cp.exists():
        combo_path = cp
        break

# If still not found, raise a clear error
if combo_path is None:
    raise FileNotFoundError(
        "Could not find 'combined_runhistory.json'. Please ensure it exists in:\n"
        f"  - {cwd / 'combined_runhistory.json'}\n"
        f"  - {cwd / 'surrogate' / 'combined_runhistory.json'}\n"
        "Or update the search paths in this cell."
    )

project_root = combo_path.parent.parent if combo_path.name == 'combined_runhistory.json' else cwd
internal_metrics_dir = project_root / 'internal_metrics'
print('Using combined_runhistory.json at:', combo_path)
print('Using internal metrics dir:', internal_metrics_dir)

Using combined_runhistory.json at: /home/E2ETune-AI4DB/surrogate/combined_runhistory.json
Using internal metrics dir: /home/E2ETune-AI4DB/internal_metrics


In [2]:
bench_patterns = {
    'job': re.compile(r"job_(\d+)_hebo_output"),
    'tpch': re.compile(r"tpch_(\d+)_hebo_output"),
    'tpcds': re.compile(r"tpcds_(\d+)_hebo_output"),
    'ssb': re.compile(r"ssb_(\d+)_hebo_output"),
    # 'tpcc': re.compile(r"sample_tpcc_config(\d+)_hebo_output"),
    # 'smallbank': re.compile(r"sample_smallbank_config(\d+)_hebo_output"),
    # 'twitter': re.compile(r"sample_twitter_config(\d+)_hebo_output"),
}

def parse_bench_and_idx(source_path: str):
    p = Path(source_path)
    parts = [str(x) for x in p.parts]
    bench = None
    for b in ['job','tpch','tpcds','ssb']:
        if b in parts:
            bench = b
            break
    if bench is None:
        return None

    match = bench_patterns[bench].search(source_path)
    if not match:
        return None
    idx = int(match.group(1))
    return bench, idx

def metrics_path_for(bench: str, idx: int) -> Path:
    # Standard path format
    standard_path = internal_metrics_dir / bench / f"{bench}_{idx}_internal_metrics.json"
    if standard_path.exists():
        return standard_path
    
    # # Alternative path formats for different benchmarks
    # if bench == 'tpcc':
    #     alt_path = internal_metrics_dir / bench / f"sample_tpcc_config{idx}.xml_internal_metrics.json"
    #     if alt_path.exists():
    #         return alt_path
    # elif bench == 'smallbank':
    #     alt_path = internal_metrics_dir / bench / f"sample_smallbank_config{idx}.xml_internal_metrics.json"
    #     if alt_path.exists():
    #         return alt_path
    # elif bench == 'twitter':
    #     alt_path = internal_metrics_dir / bench / f"sample_twitter_config{idx}.xml_internal_metrics.json"
    #     if alt_path.exists():
    #         return alt_path
    
    return standard_path

_metrics_cache = {}

def load_internal_metrics(bench: str, idx: int) -> dict:
    key = (bench, idx)
    if key in _metrics_cache:
        return _metrics_cache[key]
    mp = metrics_path_for(bench, idx)
    if not mp.exists():
        return {}
    try:
        with open(mp, 'r') as f:
            raw = json.load(f)
        flat = pd.json_normalize(raw, sep='__')
        metrics = flat.to_dict(orient='records')[0] if len(flat) else {}
    except Exception:
        metrics = {}
    _metrics_cache[key] = metrics
    return metrics


In [3]:
# Workload features and query plan feature loaders
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

workload_features_dir = project_root / 'workload_features'
query_plans_dir = project_root / 'query_plans'

_workload_cache = {}
_plans_cache = {}

# Path helpers
def workload_features_path_for(bench: str, idx: int) -> Path:
    return workload_features_dir / bench / f"{bench}_{idx}_features.json"


def query_plans_path_for(bench: str, idx: int) -> Path:
    return query_plans_dir / bench / f"{bench}_{idx}_plans.json"


# Loaders
def load_workload_features(bench: str, idx: int) -> dict:
    """Load simple workload-level features like size, ratios, avg lengths.
    Returns {} if missing or on error.
    """
    key = (bench, idx)
    if key in _workload_cache:
        return _workload_cache[key]
    p = workload_features_path_for(bench, idx)
    if not p.exists():
        _workload_cache[key] = {}
        return {}
    try:
        with open(p, 'r') as f:
            data = json.load(f)
        # Ensure numeric values and safe defaults
        result = {}
        for k, v in data.items():
            try:
                result[k] = float(v)
            except Exception:
                # Keep non-numeric as-is if needed
                result[k] = v
    except Exception:
        result = {}
    _workload_cache[key] = result
    return result


# Plan vectorization
OP_RE = re.compile(r'([A-Za-z ]+)\(cost=([0-9.]+)\)')


def vectorize_plans(plan_strings: list[str]) -> dict:
    """Parse plan strings into engineered features.
    Produces operator counts/ratios, top-level cost stats, depth proxies, and binary flags.
    """
    op_counts: dict[str, int] = {}
    top_costs: list[float] = []
    depths: list[int] = []

    for s in plan_strings or []:
        ops = OP_RE.findall(s)
        if not ops:
            continue
        # Top-level operator's cost
        try:
            top_costs.append(float(ops[0][1]))
        except Exception:
            pass
        # Simple proxy for depth: parentheses count
        depths.append(s.count('('))
        for name, cost in ops:
            key = name.strip().lower().replace(' ', '_')
            op_counts[key] = op_counts.get(key, 0) + 1

    total_ops = sum(op_counts.values()) or 1
    ratios = {f'plan__ratio__{k}': (v / total_ops) for k, v in op_counts.items()}

    agg = {
        'plan__cost_mean': float(np.mean(top_costs)) if top_costs else 0.0,
        'plan__cost_std': float(np.std(top_costs)) if top_costs else 0.0,
        'plan__cost_min': float(np.min(top_costs)) if top_costs else 0.0,
        'plan__cost_max': float(np.max(top_costs)) if top_costs else 0.0,
        'plan__depth_mean': float(np.mean(depths)) if depths else 0.0,
        'plan__depth_max': float(np.max(depths)) if depths else 0.0,
        'plan__has_index_scan': int('index_scan' in op_counts),
        'plan__has_seq_scan': int('seq_scan' in op_counts),
        'plan__has_sort': int('sort' in op_counts),
        'plan__has_aggregate': int('aggregate' in op_counts),
        'plan__has_hash_join': int('hash_join' in op_counts),
        'plan__has_nested_loop': int('nested_loop' in op_counts),
        'plan__has_merge_join': int('merge_join' in op_counts),
        'plan__has_gather': int('gather' in op_counts),
        'plan__has_gather_merge': int('gather_merge' in op_counts),
        'plan__has_bitmap_scan': int('bitmap_heap_scan' in op_counts or 'bitmap_index_scan' in op_counts),
    }
    counts = {f'plan__count__{k}': v for k, v in op_counts.items()}
    return {**counts, **ratios, **agg}


def load_query_plan_features(bench: str, idx: int) -> dict:
    """Load and parse query plan strings into engineered features.
    Returns {} if missing or on error.
    """
    key = (bench, idx)
    if key in _plans_cache:
        return _plans_cache[key]
    p = query_plans_path_for(bench, idx)
    if not p.exists():
        _plans_cache[key] = {}
        return {}
    try:
        with open(p, 'r') as f:
            data = json.load(f)
        plans = data.get('query_plans', [])
        feats = vectorize_plans(plans)
    except Exception:
        feats = {}
    _plans_cache[key] = feats
    return feats

In [ ]:
def select_config_dict(rec: dict):
    if 'config' in rec and isinstance(rec['config'], dict):
        return rec['config']
    else:
        return None


def select_cost_value(rec: dict):
    if 'cost' in rec:
        v = rec.get('cost')
        if isinstance(v, (int, float)) or (isinstance(v, list) and len(v) == 1 and isinstance(v[0], (int, float))):
            cost = float(v[0]) if isinstance(v, list) else float(v)
            return abs(cost)  # Make all costs positive
    else:
        return None

with open(combo_path, 'r') as f:
    combo = json.load(f)
records = combo.get('records', [])
print('Loaded records:', len(records))

feature_rows = []
targets = []

skipped_no_cost = 0
skipped_no_config = 0
skipped_no_metrics = 0

for rec in records:
    src = rec.get('__source', '')
    parsed = parse_bench_and_idx(src)
    if not parsed:
        skipped_no_metrics += 1
        continue
    bench, idx = parsed
    metrics = load_internal_metrics(bench, idx)
    workload = load_workload_features(bench, idx)
    plan_feats = load_query_plan_features(bench, idx)
    config = select_config_dict(rec)
    cost = select_cost_value(rec)

    if cost is None:
        skipped_no_cost += 1
        continue
    if config is None:
        skipped_no_config += 1
        continue

    config_flat = pd.json_normalize(config, sep='__').to_dict(orient='records')[0] if isinstance(config, dict) else {}
    row = {}
    row.update({f'cfg__{k}': v for k, v in config_flat.items()})
    row.update({f'metrics__{k}': v for k, v in metrics.items()})
    row.update({f'workload__{k}': v for k, v in (workload or {}).items()})
    # plan feature keys already prefixed as plan__*
    row.update(plan_feats or {})
    row['bench'] = bench
    row['workload_idx'] = idx

    feature_rows.append(row)
    targets.append(cost)

print('Rows collected:', len(feature_rows))
print('Skipped (no cost):', skipped_no_cost)
print('Skipped (no config):', skipped_no_config)
print('Skipped (no metrics parse):', skipped_no_metrics)

df = pd.DataFrame(feature_rows)
print('Raw feature shape:', df.shape)
try:
    print('Example columns (first 10):', list(df.columns)[:10])
except Exception:
    pass

df.head(3)

In [ ]:
# # Per-workload target normalization
# normalization = 'minmax'  # options: 'zscore', 'minmax'

# # Build group key (bench + workload_idx)
grp = df['bench'].astype(str) + '_' + df['workload_idx'].astype(str)
# cost_series = pd.Series(targets, name='cost')
# cost_series_log = np.log1p(pd.Series(targets, name='cost'))
# def _zscore(vals: pd.Series):
#     s = float(vals.std(ddof=0))
#     if s == 0 or np.isnan(s):
#         return vals * 0.0
#     m = float(vals.mean())
#     return (vals - m) / s

# def _minmax(vals: pd.Series):
#     mn = float(vals.min()); mx = float(vals.max())
#     if mx == mn:
#         return vals * 0.0
#     return (vals - mn) / (mx - mn)

# if normalization == 'zscore':
#     norm_series = cost_series.groupby(grp).transform(_zscore)
# else:
#     norm_series = cost_series.groupby(grp).transform(_minmax)

# y_norm = norm_series.values.astype(float)
# print(f"Per-workload normalization: {normalization}")
# print('y_norm stats -> mean:', float(np.nanmean(y_norm)), 'std:', float(np.nanstd(y_norm)))


cost_series_log = np.log1p(pd.Series(targets, name='cost'))

# Now apply MinMax on the LOG values
def _minmax(vals: pd.Series):
    mn = float(vals.min())
    mx = float(vals.max())
    if mx == mn:
        return vals * 0.0
    return (vals - mn) / (mx - mn)

# Use the log series for grouping and transformation
norm_series = cost_series_log.groupby(grp).transform(_minmax)

y_norm = norm_series.values.astype(float)
print(f"Applied Log1p -> MinMax Normalization")

In [ ]:
# Format features: one-hot categorical,split train/test
# Identify feature columns and target
y_raw = np.array(targets)
try:
    y = y_norm
except NameError:
    y = y_raw

X = df.copy()

# Drop grouping-only columns to prevent leakage
for col in ['bench', 'workload_idx']:
    if col in X.columns:
        X = X.drop(columns=[col])

# Normalize knob columns using ranges from knob_config/knob_config.json
import json

# Load knob ranges (use project_root if available)
try:
    with open(project_root / 'knob_config/knob_config.json', 'r') as f:
        knob_ranges = json.load(f)
except Exception:
    with open('knob_config/knob_config.json', 'r') as f:
        knob_ranges = json.load(f)

def _normalize_knob(orig_key, val):
    rng = knob_ranges.get(orig_key)
    if rng is None:
        # If knob not in ranges, pass through numeric value
        return float(val) if pd.notnull(val) else 0.0
    mn = float(rng.get('min', 0.0))
    mx = float(rng.get('max', mn))
    if not pd.notnull(val):
        return 0.0
    v = float(val)
    return 0.0 if mx == mn else (v - mn) / (mx - mn)

# Apply normalization to all cfg__* columns
cfg_cols = [c for c in X.columns if c.startswith('cfg__')]
for c in cfg_cols:
    orig = c.replace('cfg__', '')
    X[c] = pd.to_numeric(X[c], errors='coerce')
    X[c] = X[c].apply(lambda v: _normalize_knob(orig, v))

# Convert categorical columns (object dtype) to dummies
categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
X = pd.get_dummies(X, columns=categorical_cols, dummy_na=True)

# Optional: ensure numeric type
for col in X.columns:
    # Convert booleans to ints
    if X[col].dtype == bool:
        X[col] = X[col].astype(int)
    # Convert any non-numeric to string dummies already; remaining should be numeric
    if not np.issubdtype(X[col].dtype, np.number):
        try:
            X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
        except Exception:
            pass

print('Final feature shape:', X.shape)

In [ ]:
# Group-aware train/test split (hold out entire workloads)
from sklearn.model_selection import GroupShuffleSplit
groups = df['bench'].astype(str) + '_' + df['workload_idx'].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

# from sklearn.model_selection import train_test_split
# train_idx, test_idx = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)

# Train-only min–max normalization for metrics/workload/plan numeric columns
metrics_cols = [c for c in X.columns if c.startswith('metrics__')]
workload_cols = [c for c in X.columns if c.startswith('workload__')]
# Normalize plan numeric features but keep binary flags (plan__has_*) as 0/1
plan_norm_cols = [c for c in X.columns if c.startswith('plan__') and not c.startswith('plan__has_')]

norm_cols = metrics_cols + workload_cols + plan_norm_cols
if len(norm_cols) > 0:
    # Ensure numeric
    for c in norm_cols:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    train_subset = X.iloc[train_idx][norm_cols]
    mn = train_subset.min()
    mx = train_subset.max()
    rng = mx - mn
    rng = rng.replace(0, np.nan)
    X.loc[:, norm_cols] = (X[norm_cols] - mn) / rng
    X.loc[:, norm_cols] = X[norm_cols].fillna(0.0).clip(0.0, 1.0)
    print('Applied train-only min–max normalization to metrics__/workload__/plan__* (excluding flags).')

x_train, x_test = X.values[train_idx], X.values[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print('Group-aware split:', x_train.shape, x_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Model training and evaluation

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(x_train, y_train)

# Make predictions
y_train_pred = rf_model.predict(x_train)
y_test_pred = rf_model.predict(x_test)

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'Train MSE: {train_mse:.6f}, R²: {train_r2:.6f}')
print(f'Test MSE: {test_mse:.6f}, R²: {test_r2:.6f}')

# Save model and feature information for inference
try:
    out_dir = project_root / 'surrogate'
except NameError:
    out_dir = Path('surrogate')

out_dir.mkdir(exist_ok=True)

# Save the trained model
joblib.dump(rf_model, out_dir / 'cost_model.pkl')
print(f'Model saved to: {out_dir / "cost_model.pkl"}')

# Save feature names
with open(out_dir / 'best_r2_features.txt', 'w') as f:
    for feat in X.columns:
        f.write(f'{feat}\n')

# Save feature scaling parameters (for inference normalization)
scaler_info = {}
if len(norm_cols) > 0:
    train_subset = X.iloc[train_idx][norm_cols]
    scaler_info = {
        'min': train_subset.min().to_dict(),
        'max': train_subset.max().to_dict()
    }
    
with open(out_dir / 'feature_scaler.json', 'w') as f:
    json.dump(scaler_info, f, indent=2)

# Save target transformation info
target_info = {'target': 'y_log', 'method': 'log1p_minmax_per_group'}
with open(out_dir / 'target_transform.json', 'w') as f:
    json.dump(target_info, f, indent=2)

print(f'Feature scaler and target info saved to: {out_dir}')

In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

# Inference helpers: predict using trained cost_model.pkl with train-only scaling
try:
    out_dir = project_root / 'surrogate'
except NameError:
    out_dir = Path('surrogate')


def _load_feature_list(model, out_dir: Path):
    feat_list = []
    try:
        if hasattr(model, 'feature_names_in_'):
            feat_list = list(model.feature_names_in_)
    except Exception:
        pass
    for p in [out_dir / 'best_r2_features.txt', out_dir / 'cost_model_features.txt']:
        if not feat_list and p.exists():
            try:
                with open(p, 'r') as f:
                    feat_list = [line.strip() for line in f if line.strip()]
                break
            except Exception:
                pass
    return feat_list


def _align_columns_df(X: pd.DataFrame, model, out_dir: Path) -> pd.DataFrame:
    feat_list = _load_feature_list(model, out_dir)
    if not feat_list:
        return X
    for c in feat_list:
        if c not in X.columns:
            X[c] = 0.0
    return X[feat_list]


def _load_scaler(out_dir: Path):
    scaler_path = out_dir / 'feature_scaler.json'
    try:
        with open(scaler_path, 'r') as f:
            scaler = json.load(f)
        if 'min' in scaler and 'max' in scaler:
            return scaler['min'], scaler['max']
        return scaler.get('feature_min', {}), scaler.get('feature_max', {})
    except Exception:
        return {}, {}


def _apply_train_minmax_row(row: dict, mn_map: dict, mx_map: dict) -> dict:
    if not mn_map or not mx_map:
        return row
    out = dict(row)
    for k, v in row.items():
        if k.startswith('metrics__') or k.startswith('workload__') or (
            k.startswith('plan__') and not k.startswith('plan__has_')
        ):
            mn = mn_map.get(k)
            mx = mx_map.get(k)
            if mn is None or mx is None:
                continue
            try:
                val = float(v)
            except Exception:
                val = 0.0
            rng = mx - mn
            if rng <= 0:
                out[k] = 0.0
            else:
                scaled = (val - mn) / rng
                if not np.isfinite(scaled):
                    scaled = 0.0
                out[k] = float(np.clip(scaled, 0.0, 1.0))
    return out


def _load_knob_ranges():
    try:
        with open(project_root / 'knob_config/knob_config.json', 'r') as f:
            return json.load(f)
    except Exception:
        try:
            with open('knob_config/knob_config.json', 'r') as f:
                return json.load(f)
        except Exception:
            return {}


def _normalize_knob(orig_key, val, knob_ranges):
    rng = knob_ranges.get(orig_key)
    if rng is None:
        try:
            return float(val) if pd.notnull(val) else 0.0
        except Exception:
            return 0.0
    mn = float(rng.get('min', 0.0))
    mx = float(rng.get('max', mn))
    if not pd.notnull(val):
        return 0.0
    v = float(val)
    return 0.0 if mx == mn else (v - mn) / (mx - mn)


def build_row(
    config_dict: dict,
    metrics_dict: dict | None = None,
    workload_feats: dict | None = None,
    plan_feats: dict | None = None,
    plan_strings: list[str] | None = None,
) -> dict:
    """Build a single feature row.
    - If plan_feats dict is provided, use it.
    - Else if plan_strings list is provided, vectorize into engineered features.
    """
    knob_ranges = _load_knob_ranges()
    cfg_flat = pd.json_normalize(config_dict or {}, sep='__').to_dict(orient='records')
    cfg_flat = cfg_flat[0] if cfg_flat else {}
    cfg_norm = {f'cfg__{k}': _normalize_knob(k, v, knob_ranges) for k, v in cfg_flat.items()}
    row = {}
    row.update(cfg_norm)
    mflat = pd.json_normalize(metrics_dict or {}, sep='__').to_dict(orient='records')
    mflat = mflat[0] if mflat else {}
    row.update({f'metrics__{k}': v for k, v in mflat.items()})
    wl_numeric = {}
    for k, v in (workload_feats or {}).items():
        try:
            wl_numeric[k] = float(v)
        except Exception:
            wl_numeric[k] = v
    row.update({f'workload__{k}': v for k, v in wl_numeric.items()})
    feats = {}
    if isinstance(plan_feats, dict):
        feats = plan_feats
    elif plan_strings is not None:
        try:
            # Uses vectorize_plans defined earlier in the notebook
            feats = vectorize_plans(plan_strings)
        except Exception:
            feats = {}
    for k, v in feats.items():
        row[k] = v
    return row


def predict_cost(
    config_dict: dict,
    metrics_dict: dict | None = None,
    workload_feats: dict | None = None,
    plan_feats: dict | None = None,
    plan_strings: list[str] | None = None,
    model_path: str | Path | None = None,
    force_log: bool = True,
) -> float:
    """Predict cost given raw config, metrics, workload features, and either plan_feats or raw plan_strings.
    If force_log=True (default), invert predictions from log-space via expm1.
    """
    if model_path is None:
        model_path = out_dir / 'cost_model.pkl'
    model = joblib.load(model_path)
    mn_map, mx_map = _load_scaler(out_dir)
    row = build_row(config_dict, metrics_dict, workload_feats, plan_feats, plan_strings)
    row = _apply_train_minmax_row(row, mn_map, mx_map)
    df = pd.DataFrame([row])
    cat_cols = [c for c in df.columns if df[c].dtype == 'object']
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, dummy_na=True)
    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number):
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    X = _align_columns_df(df.copy(), model, out_dir)
    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], 0.0).fillna(0.0)
    preds = model.predict(X.values.astype(np.float64))
    # Invert log by default since cost_model was trained on log1p(cost)
    if force_log:
        return float(np.expm1(preds[0]))
    # Fallback: use artifact-driven inversion if available
    try:
        with open(out_dir / 'target_transform.json', 'r') as f:
            tinfo = json.load(f)
        if str(tinfo.get('target')) == 'y_log':
            return float(np.expm1(preds[0]))
    except Exception:
        pass
    return float(preds[0])

# Example usage (fill with actual values):
config_example =     {
        "shared_buffers": 12,
        "work_mem": 16,
        "maintenance_work_mem": 13,
        "effective_cache_size": 14,
        "max_connections": 13,
        "wal_buffers": 7,
        "checkpoint_completion_target": 1,
        "checkpoint_timeout": 19,
        "effective_io_concurrency": 16,
        "join_collapse_limit": 1,
        "from_collapse_limit": 1,
        "bgwriter_delay": 14,
        "bgwriter_lru_multiplier": 18,
        "default_statistics_target": 14,
        "max_parallel_workers_per_gather": 7
    }
metrics_example = json.load(open(project_root / 'inference_test/analysis_output/job/job_internal_metrics.json'))
wl_feats_example = json.load(open(project_root / 'inference_test/analysis_output/job/job_features.json'))
plan_strings_example = json.load(open(project_root / 'inference_test/analysis_output/job/job_plans.json'))["query_plans"]
print(predict_cost(config_example, metrics_example, wl_feats_example, plan_strings=plan_strings_example, model_path=out_dir / 'cost_model.pkl'))


55.74605151438559


# Inference Usage
- Ensure `surrogate/cost_model.pkl`, `surrogate/feature_scaler.json`, and optionally `surrogate/best_r2_features.txt` exist.
- Provide a config dict (raw knob values), internal metrics, workload features, and optional engineered plan features.
- Call `predict_cost(config_dict, metrics_dict, workload_feats, plan_feats, model_path, force_log=True)` to get the predicted cost.
- Knobs are normalized using ranges from `knob_config/knob_config.json`; metrics/workload/plan numeric features are min–max scaled using train-set stats.
- Predictions are inverted from log-space by default (`force_log=True`) since `cost_model.pkl` was trained on `log1p(cost)`. Set `force_log=False` to rely on `target_transform.json` or return native model scale.
- If training used `y_norm` (log1p → per-group min–max), full inversion at inference requires the same group stats; for unseen groups, normalized outputs cannot be safely un-normalized.